# 11 — Giá và diễn biến thị trường

Notebook đầu tiên của Track 1. Bốn thứ bạn sẽ dùng lại ở mọi notebook sau:

1. Lấy giá **nhiều mã trong một request** và tách chúng ra đúng cách
2. `interval` — gộp nhóm chạy ở server, không phải ở máy bạn
3. **Giá điều chỉnh vs giá thô** — và vì sao chọn nhầm làm hỏng mọi backtest
4. So hiệu suất giữa các mã: chuẩn hoá về gốc 100

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay, nen, nhan_don_vi, thanh_doi_mau

ap_dung_theme()
client = finlens.client()

HOM_NAY = hom_nay(client)
print(f"Phiên tham chiếu: {HOM_NAY}")

Phiên tham chiếu: 2026-08-11


## 1 · Chỉ số: bốn mã, và hai mã README ghi sai

Thị trường Việt Nam có năm chỉ số chính. Tên của chúng **không** suy được từ
tên sàn — README của thư viện ghi `HNX-INDEX` và `UPCOM-INDEX`, nhưng API nhận
`HNXINDEX` và `UPINDEX`. Hai tên có gạch nối trả về `InvalidSymbolError`.

In [2]:
CHI_SO = ["VNINDEX", "VN30", "HNXINDEX", "HNX30", "UPINDEX"]

chi_so = client.eod.index.ohlcv(CHI_SO, start=lui_ngay(HOM_NAY, nam=1))
print(f"{chi_so.shape[0]:,} dòng · đơn vị giá: {chi_so.attrs['finlens']['units']['close']}")

chi_so.groupby("symbol", observed=True).agg(
    phien_dau=("date", "min"),
    phien_cuoi=("date", "max"),
    dong_cua=("close", "last"),
).assign(dong_cua=lambda d: d["dong_cua"].round(2))

1,250 dòng · đơn vị giá: index_point


,phien_dau,phien_cuoi,dong_cua
symbol,,,
HNX30,2025-08-11,2026-08-11,466.92
HNXINDEX,2025-08-11,2026-08-11,288.50
UPINDEX,2025-08-11,2026-08-11,127.50
VN30,2025-08-11,2026-08-11,1925.82
VNINDEX,2025-08-11,2026-08-11,1776.50


⚠️ Đơn vị là **`index_point`**, không phải `kVND`. VNINDEX ở `1776.5` là 1.776,5
điểm — không phải 1.776.500 đồng. Đây là lý do chỉ số nằm ở namespace riêng
`client.eod.index` chứ không trộn chung với cổ phiếu.

## 2 · So hiệu suất: chuẩn hoá về gốc 100

Vẽ thẳng `close` của năm chỉ số lên một khung là vô nghĩa — UPINDEX quanh 127
điểm còn VN30 quanh 1.925, đường UPINDEX sẽ nằm bẹp dưới đáy.

Và **không** giải quyết bằng hai trục y. Hai trục y là lỗi biểu đồ phổ biến
nhất: độ cao tương đối giữa hai đường khi đó do người vẽ chọn thang, không phải
do dữ liệu quyết định. Cách đúng là **đưa về cùng một thang**: lấy phiên đầu
làm gốc 100.

In [3]:
def ve_goc_100(df: pd.DataFrame, cot_khoa: str = "symbol", cot_gia: str = "close") -> pd.DataFrame:
    """Chuẩn hoá mỗi chuỗi về 100 tại phiên đầu tiên của chính nó."""
    df = df.sort_values([cot_khoa, "date"])
    goc = df.groupby(cot_khoa, observed=True)[cot_gia].transform("first")
    return df.assign(chi_so_100=df[cot_gia] / goc * 100)


hieu_suat = ve_goc_100(chi_so)

duong(
    hieu_suat,
    x="date",
    y="chi_so_100",
    theo="symbol",
    tieu_de="Hiệu suất năm chỉ số, 12 tháng gần nhất",
    phu_de="Chuẩn hoá về 100 tại phiên đầu — cùng một thang, một trục y",
    nhan_y="chỉ số (gốc = 100)",
)

Đọc được ngay: chỉ số nào dẫn dắt, chỉ số nào tụt lại, và khoảng cách giữa
chúng nở ra hay hẹp lại vào lúc nào. Không thông tin nào trong đó lấy được từ
một biểu đồ hai trục y.

In [4]:
tong_ket = (
    hieu_suat.groupby("symbol", observed=True)["chi_so_100"]
    .last()
    .sub(100)
    .round(2)
    .reset_index()
    .rename(columns={"chi_so_100": "thay_doi_pct"})
)

thanh_doi_mau(
    tong_ket,
    x="symbol",
    y="thay_doi_pct",
    tieu_de="Thay đổi 12 tháng theo chỉ số",
    phu_de="Xanh tăng · đỏ giảm — con số in trên cột vì màu chỉ là kênh phụ",
    nhan_y="%",
    dinh_dang_nhan="{:+.1f}%",
)

## 3 · `interval` — gộp nhóm chạy ở server

`interval` nhận `1d`, `1w`, `1mo`, `3mo`, `6mo`, `1y`. Việc gộp chạy **ở
server**, nên bạn tải về ít dữ liệu hơn chứ không phải tải hết rồi `resample`.

⚠️ `interval="1M"` bị từ chối vì nhập nhằng giữa *một tháng* và *một phút*.
Dùng `1mo` hoặc `1min`.

In [5]:
ma = "HPG"
bat_dau = lui_ngay(HOM_NAY, nam=3)

for buoc in ["1d", "1w", "1mo", "3mo"]:
    d = client.eod.stock.ohlcv(ma, start=bat_dau, interval=buoc)
    print(f"  interval={buoc:<5} → {len(d):>4} dòng")

try:
    client.eod.stock.ohlcv(ma, start=bat_dau, interval="1M")
except finlens.InvalidIntervalError as e:
    print(f"\n  interval='1M'  → {type(e).__name__}: {e}")

  interval=1d    →  747 dòng


  interval=1w    →  156 dòng


  interval=1mo   →   37 dòng


  interval=3mo   →   13 dòng

  interval='1M'  → InvalidIntervalError: [FL_VALIDATION_INTERVAL_AMBIGUOUS] Interval '1M' nhập nhằng: có thể là một phút hoặc một tháng. Dùng '1min' cho một phút, hoặc '1mo' cho một tháng. (request_id=993e7503480049e5a5a845275e206b5e) -> https://docs.finlens.vn/python-sdk/errors/FL_VALIDATION_INTERVAL_AMBIGUOUS


Nến tuần của HPG ba năm — cùng dữ liệu, ít nhiễu hơn nến ngày:

In [6]:
tuan = client.eod.stock.ohlcv(ma, start=bat_dau, interval="1w")

nen(
    tuan,
    tieu_de=f"{ma} — nến tuần, 3 năm",
    phu_de="Giá và khối lượng là hai hàng subplot, không phải hai trục y chồng nhau",
    nhan_gia=nhan_don_vi(tuan, "close"),
)

## 4 · Giá điều chỉnh vs giá thô — chỗ hỏng backtest

Mặc định `adjusted=True`: giá đã điều chỉnh cho cổ tức, cổ phiếu thưởng, chia
tách. Đó là giá bạn cần để **đo lợi suất**.

`adjusted=False` cho giá thô đúng như bảng điện phiên hôm đó. Đó là giá bạn cần
để **đối chiếu với một bản ghi lịch sử** — sao kê tài khoản, một bài báo cũ.

Chọn nhầm không ném lỗi. Nó chỉ làm mọi lợi suất bạn tính bị sai.

In [7]:
MA_CO_TUC = "VNM"  # trả cổ tức tiền mặt đều đặn

dieu_chinh = client.eod.stock.ohlcv(MA_CO_TUC, start=lui_ngay(HOM_NAY, nam=2))
gia_tho = client.eod.stock.ohlcv(MA_CO_TUC, start=lui_ngay(HOM_NAY, nam=2), adjusted=False)

print(f"price_basis: {dieu_chinh.attrs['finlens']['price_basis']} · {gia_tho.attrs['finlens']['price_basis']}")

so_sanh = dieu_chinh.merge(gia_tho, on=["symbol", "date"], suffixes=("_dc", "_tho"))
lech = so_sanh[so_sanh["close_dc"].round(2) != so_sanh["close_tho"].round(2)]

print(f"\n{MA_CO_TUC}: {len(lech):,}/{len(so_sanh):,} phiên có giá điều chỉnh khác giá thô")
print(f"Phiên đầu kỳ — điều chỉnh {so_sanh['close_dc'].iloc[0]:,.2f} vs thô {so_sanh['close_tho'].iloc[0]:,.2f}")

ls_dc = so_sanh["close_dc"].iloc[-1] / so_sanh["close_dc"].iloc[0] - 1
ls_tho = so_sanh["close_tho"].iloc[-1] / so_sanh["close_tho"].iloc[0] - 1
print(f"\nLợi suất 2 năm tính trên giá điều chỉnh: {ls_dc:+.2%}")
print(f"Lợi suất 2 năm tính trên giá thô:        {ls_tho:+.2%}")
print(f"Chênh lệch:                              {abs(ls_dc - ls_tho):.2%} điểm phần trăm")

price_basis: adjusted · raw

VNM: 465/498 phiên có giá điều chỉnh khác giá thô
Phiên đầu kỳ — điều chỉnh 62.53 vs thô 73.00

Lợi suất 2 năm tính trên giá điều chỉnh: -0.53%
Lợi suất 2 năm tính trên giá thô:        -14.79%
Chênh lệch:                              14.27% điểm phần trăm


Chênh lệch đó chính là phần cổ tức bạn đã nhận nhưng giá thô không ghi lại.
Một chiến lược backtest trên giá thô sẽ báo lỗ ở đúng những mã trả cổ tức tốt
nhất.

Hai đường giá đặt cạnh nhau — khoảng hở giữa chúng nở ra sau mỗi lần chia:

In [8]:
doi_chieu = pd.concat(
    [
        dieu_chinh.assign(loai="đã điều chỉnh"),
        gia_tho.assign(loai="giá thô"),
    ]
)

duong(
    doi_chieu,
    x="date",
    y="close",
    theo="loai",
    tieu_de=f"{MA_CO_TUC} — giá điều chỉnh so với giá thô",
    phu_de="Khoảng hở giữa hai đường là cổ tức và quyền đã trả",
    nhan_y=nhan_don_vi(dieu_chinh, "close"),
)

## 5 · Nhiều mã trong một request

`max_symbols_per_request` của gói này là 100. Vượt số đó **không phải lỗi của
bạn** — thư viện tự chia lô và gọi song song. Bạn cứ truyền cả danh sách.

In [9]:
import time

ma_hose = client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()
print(f"HOSE có {len(ma_hose)} cổ phiếu (đã bỏ chứng chỉ quỹ)")

t0 = time.perf_counter()
toan_san = client.eod.stock.ohlcv(ma_hose, start=lui_ngay(HOM_NAY, thang=1), refresh=True)
mat = time.perf_counter() - t0

print(f"{len(ma_hose)} mã · {len(toan_san):,} dòng · {mat:.1f} giây")
print(f"truncated: {toan_san.attrs['finlens']['truncated']}  ← luôn kiểm, max_rows là {client.limits()['max_rows']:,}")

HOSE có 407 cổ phiếu (đã bỏ chứng chỉ quỹ)

407 mã · 8,859 dòng · 16.2 giây
truncated: False  ← luôn kiểm, max_rows là 100,000


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


### Top tăng / giảm một tháng

Bảng xếp hạng đầu tiên — và bài học đi kèm: xếp hạng trên **toàn bộ** 407 mã
cho ra một danh sách đứng đầu bởi các mã gần như không giao dịch. Lọc thanh
khoản không phải bước làm đẹp, nó là bước làm cho kết quả có nghĩa.

In [10]:
thay_doi = (
    toan_san.sort_values(["symbol", "date"])
    .groupby("symbol", observed=True)
    .agg(
        gia_dau=("close", "first"),
        gia_cuoi=("close", "last"),
        kl_bq=("volume", "mean"),
        gia_bq=("close", "mean"),
    )
)
thay_doi["thay_doi_pct"] = (thay_doi["gia_cuoi"] / thay_doi["gia_dau"] - 1) * 100
# Giá tính bằng nghìn VND → nhân 1.000 để ra VND. Bỏ số 1.000 này là lệch ba chữ số.
thay_doi["gtgd_ty"] = thay_doi["gia_bq"] * thay_doi["kl_bq"] * 1_000 / 1e9

thanh_khoan = thay_doi[thay_doi["gtgd_ty"] >= 10].reset_index()
print(f"Còn {len(thanh_khoan)}/{len(thay_doi)} mã có GTGD bình quân ≥ 10 tỷ/phiên")

top = pd.concat(
    [
        thanh_khoan.nlargest(8, "thay_doi_pct"),
        thanh_khoan.nsmallest(8, "thay_doi_pct"),
    ]
)

from finlens_examples import bar_ngang

bar_ngang(
    top,
    nhan="symbol",
    gia_tri="thay_doi_pct",
    tieu_de="Top tăng và giảm một tháng — HOSE",
    phu_de="Chỉ tính các mã có giá trị giao dịch bình quân ≥ 10 tỷ đồng/phiên",
    nhan_x="% thay đổi",
    dinh_dang_nhan="{:+.1f}%",
)

Còn 112/403 mã có GTGD bình quân ≥ 10 tỷ/phiên


So sánh: nếu bỏ bộ lọc thanh khoản thì danh sách dẫn đầu trông thế nào?

In [11]:
khong_loc = thay_doi.reset_index().nlargest(8, "thay_doi_pct")[
    ["symbol", "thay_doi_pct", "gtgd_ty"]
]
print("Top tăng KHÔNG lọc thanh khoản:\n")
print(khong_loc.round(2).to_string(index=False))
print(f"\nGTGD bình quân của nhóm này: {khong_loc['gtgd_ty'].median():.2f} tỷ/phiên")
print(f"GTGD bình quân của nhóm đã lọc: {top.nlargest(8, 'thay_doi_pct')['gtgd_ty'].median():.1f} tỷ/phiên")

Top tăng KHÔNG lọc thanh khoản:



symbol  thay_doi_pct  gtgd_ty
   HII         70.05     3.73
   VVS         30.64    18.56
   FRT         29.30    62.68
   ASP         28.02     3.37
   PGI         27.82     0.28
   SPM         23.51     0.03
   CTF         18.88     5.98
   BVH         14.61    25.58

GTGD bình quân của nhóm này: 4.85 tỷ/phiên
GTGD bình quân của nhóm đã lọc: 44.1 tỷ/phiên


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Năm chỉ số | `eod.index.ohlcv(["VNINDEX","VN30","HNXINDEX","HNX30","UPINDEX"])` |
| Nến tuần / tháng | `eod.stock.ohlcv("HPG", interval="1w")` |
| Giá đúng như bảng điện hôm đó | `eod.stock.ohlcv("VNM", adjusted=False)` |
| Cả sàn HOSE một lần | `eod.stock.ohlcv(danh_sach_407_ma)` |

**Bốn điều mang sang notebook sau:**

1. Chỉ số dùng `index_point`, cổ phiếu dùng `kVND` — hai namespace riêng vì
   thế, và `HNX-INDEX` không phải mã hợp lệ.
2. So nhiều chuỗi khác thang thì **chuẩn hoá về gốc 100**, không bao giờ dùng
   hai trục y.
3. `adjusted=True` để đo lợi suất, `adjusted=False` để đối chiếu bản ghi lịch
   sử. Chọn nhầm không có cảnh báo nào.
4. Mọi bảng xếp hạng cần một bộ lọc thanh khoản, nếu không nó xếp hạng nhiễu.

---

**Tiếp theo:** [`12_ban_do_nganh_icb.ipynb`](12_ban_do_nganh_icb.ipynb) — cây
ngành ICB, sức mạnh ngành và độ rộng thị trường.